# GNN Conjecture Mining on BME-MathGraph

Demonstrates structure-hole detection and GNN link prediction for
cross-domain conjecture generation (Section 6 of the paper).

In [ ]:
import numpy as np
import networkx as nx

G = nx.DiGraph()

nodes = {
    'D_M44': 'META', 'D_GExp': 'GEXP', 'T_ClassicalFL': 'NT',
    'D_Hitchin': 'GEO', 'T_GL2_SFL': 'NT', 'I_GEXP_Motivation': 'META',
    'L_SFL_Split': 'NT', 'L_SFL_Elliptic': 'GEO',
    'T_Phi_Functor': 'MART', 'C_Entropy': 'MART',
    'T_Hitchin_Support': 'GEO', 'C_Stochastic_Hitchin': 'GEO',
}
for n, d in nodes.items():
    G.add_node(n, domain=d)

edges = [
    ('D_M44', 'T_GL2_SFL', 0.8), ('D_GExp', 'T_GL2_SFL', 0.5),
    ('T_ClassicalFL', 'I_GEXP_Motivation', 0.4),
    ('I_GEXP_Motivation', 'T_GL2_SFL', 0.2),
    ('D_Hitchin', 'T_GL2_SFL', 0.4),
    ('L_SFL_Split', 'T_GL2_SFL', 0.6),
    ('L_SFL_Elliptic', 'T_GL2_SFL', 0.6),
    ('D_GExp', 'T_Phi_Functor', 0.7),
    ('T_GL2_SFL', 'T_Phi_Functor', 0.5),
    ('T_Phi_Functor', 'C_Entropy', 0.6),
    ('T_GL2_SFL', 'C_Stochastic_Hitchin', 0.5),
    ('D_Hitchin', 'T_Hitchin_Support', 0.7),
    ('C_Stochastic_Hitchin', 'T_Hitchin_Support', 0.8),
]
for u, v, w in edges:
    G.add_edge(u, v, weight=w, type='uses')

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

In [ ]:
def constraint_coefficient(G, node):
    neighbors = list(G.neighbors(node))
    if not neighbors:
        return 1.0
    total = sum(G[node][n]['weight'] for n in neighbors)
    probs = {n: G[node][n]['weight'] / total if total > 0 else 0 for n in neighbors}
    C = 0.0
    for j in neighbors:
        s = probs[j]
        for k in neighbors:
            if G.has_edge(j, k):
                s += probs[j] * probs[k]
        C += s ** 2
    return C

constraints = {n: constraint_coefficient(G, n) for n in G.nodes()}
sorted_c = sorted(constraints.items(), key=lambda x: x[1])
print('\nConstraint coefficients (low = high structural hole):')
for n, c in sorted_c:
    print(f'  {n:25s} C={c:.4f}  domain={G.nodes[n]["domain"]}')

In [ ]:
threshold = 0.5
hotspots = []
node_list = list(G.nodes())
for i in range(len(node_list)):
    for j in range(i+1, len(node_list)):
        n1, n2 = node_list[i], node_list[j]
        if G.nodes[n1]['domain'] != G.nodes[n2]['domain']:
            if not G.has_edge(n1, n2) and not G.has_edge(n2, n1):
                c = (constraints[n1] + constraints[n2]) / 2
                if c < threshold:
                    hotspots.append((n1, n2, c, G.nodes[n1]['domain'], G.nodes[n2]['domain']))

print(f'\n=== {len(hotspots)} Conjecture Hotspots ===')
for n1, n2, c, d1, d2 in sorted(hotspots, key=lambda x: x[2])[:5]:
    print(f'  [{d1}] {n1}  <-->  [{d2}] {n2}   C={c:.4f}')
    print(f'    Candidate: stochastic functor Phi between {n1} and {n2}')

The GNN pipeline identifies cross-domain pairs with low constraint as conjecture hotspots, candidates for new functorial transfers between deterministic Langlands and stochastic sides.